# 🚀 DL-PM25-BJ — Pipeline Huấn luyện GPU, Fine-tuning 12 Trạm & Ablation trên Google Colab

- **Dự án:** Dự báo nồng độ bụi mịn $\text{PM}_{2.5}$ tại Bắc Kinh (`beijing-pm25-deeplearning`)
- **Nhánh Git chuẩn:** `fix/BPM25_T0305_wp03-training-audit`
- **Tệp cấu hình duy nhất:** `configs/default.yaml`
- **Thư mục Backup Google Drive:** `https://drive.google.com/drive/u/0/folders/1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`
- **Google Drive Folder ID:** `1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`

## Cell 1 — Mount Google Drive & Kiểm tra GPU (T4 / L4 / A100)

Thực hiện:
1. Kết nối Google Drive tại `/content/drive`.
2. Kiểm tra GPU CUDA khả dụng (`nvidia-smi` và `torch.cuda.is_available()`).
3. Thiết lập `GDRIVE_FOLDER_ID = "1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL"` và thư mục đồng bộ cục bộ trên Google Drive.

In [ ]:
import os
from pathlib import Path
import torch
from google.colab import drive

drive.mount("/content/drive")

GDRIVE_FOLDER_ID = "1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL"
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/DL_PM25_BJ_Backup")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

print(f"[INFO] GDRIVE_FOLDER_ID = {GDRIVE_FOLDER_ID}")
print(f"[INFO] DRIVE_BACKUP_DIR = {DRIVE_BACKUP_DIR}")
print(f"[GPU] CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[GPU] Device: {torch.cuda.get_device_name(0)}")
!nvidia-smi

## Cell 2 — Đồng bộ Mã nguồn & Cài đặt Môi trường

- Clone đúng nhánh `fix/BPM25_T0305_wp03-training-audit` (hoặc giải nén từ gói mã nguồn nếu repo là private chưa cấu hình token).
- Cài đặt các gói phụ thuộc từ `requirements.txt` (giữ nguyên PyTorch CUDA mặc định của Google Colab) cùng `mlflow` và `google-api-python-client` phục vụ backup.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/<YOUR_ORG_OR_USER>/beijing-pm25-deeplearning.git"  # Cập nhật URL repo thực tế
BRANCH_NAME = "fix/BPM25_T0305_wp03-training-audit"
WORK_ROOT = Path("/content/beijing-pm25-deeplearning")
ARCHIVE_ON_DRIVE = DRIVE_BACKUP_DIR / "course_project_src.zip"

if not WORK_ROOT.exists():
    if ARCHIVE_ON_DRIVE.exists():
        print(f"[INFO] Giải nén mã nguồn từ {ARCHIVE_ON_DRIVE}...")
        !unzip -q "{ARCHIVE_ON_DRIVE}" -d /content/beijing-pm25-deeplearning
    else:
        print(f"[INFO] Clone nhánh {BRANCH_NAME}...")
        !git clone -b {BRANCH_NAME} {REPO_URL} {WORK_ROOT}

PROJECT_DIR = WORK_ROOT / "course_project"
if not PROJECT_DIR.exists():
    PROJECT_DIR = WORK_ROOT

os.chdir(PROJECT_DIR)
os.environ["MLFLOW_TRACKING_URI"] = str(PROJECT_DIR / "mlruns")
print(f"[CWD] {os.getcwd()}")
print(f"[MLFLOW] MLFLOW_TRACKING_URI = {os.environ['MLFLOW_TRACKING_URI']}")

# Giữ nguyên PyTorch CUDA mặc định của Colab, chỉ cài các gói bổ sung còn thiếu
!grep -v '^torch' requirements.txt > /tmp/req_no_torch.txt
!pip install -q -r /tmp/req_no_torch.txt mlflow google-api-python-client google-auth-httplib2

## Cell 3 — Chuẩn bị Dữ liệu (Tầng 2 & Tầng 3: 12 Trạm UCI PRSA, D=31)

- Nạp bộ 12 trạm từ Google Drive (nếu đã lưu sẵn) hoặc tải bằng `scripts/download_data.py`.
- Chạy `python -m src.data_pipeline.preprocess --config configs/default.yaml` để sinh `train.csv`, `train_dev.csv`, `dev.csv`, `test.csv` và `standard_scaler.pkl` chuẩn nhân quả 100%, không Data Leakage.

In [ ]:
from pathlib import Path
import shutil

raw_dir = Path("data/raw")
drive_raw_dir = DRIVE_BACKUP_DIR / "data_raw"

if not list(raw_dir.glob("PRSA_Data_*.csv")):
    if drive_raw_dir.exists() and list(drive_raw_dir.glob("PRSA_Data_*.csv")):
        raw_dir.mkdir(parents=True, exist_ok=True)
        for csv_p in drive_raw_dir.glob("PRSA_Data_*.csv"):
            shutil.copy2(csv_p, raw_dir / csv_p.name)
        print("[INFO] Đã copy 12 trạm từ Google Drive sang data/raw/")
    else:
        !python scripts/download_data.py

!python -m src.data_pipeline.preprocess --config configs/default.yaml
!ls -lh data/preprocess/

## Cell 4 — Huấn luyện Pre-train Đa Mô hình & 3 Seeds (`[42, 1024, 2026]`)

- Chạy đối chứng đầy đủ 4 kiến trúc chính (`lstm`, `dlinear`, `informer`, `patchtst`) trên tất cả 12 trạm qua `training_loop.py --config configs/default.yaml`.
- Thực thi trên các mốc chân trời $H \in \{24, 48, 96, 192\}$ với 3 seeds `[42, 1024, 2026]` để xuất báo cáo $\text{mean} \pm \text{std}$ và `ExperimentMetricsContract`.

In [ ]:
HORIZONS = [24, 48, 96, 192]
MODELS = ["lstm", "dlinear", "informer", "patchtst"]

for h in HORIZONS:
    print(f"\n==================== PRE-TRAIN H={h} ====================")
    models_arg = " ".join(MODELS)
    !python training_loop.py --config configs/default.yaml --horizon {h} --models {models_arg} --seeds 42 1024 2026

## Cell 5 — Fine-tuning 12 Trạm & Ablation Studies

1. Chạy `fine_tune.py --config configs/default.yaml` cho từng khung dự báo $H \in \{24, 48, 96, 192\}$ trên đủ 12 trạm.
2. Chạy `run_ablation.py` thực hiện 3 kịch bản cắt bỏ (Multivariate vs Univariate, Multi-Horizon, Informer ProbSparse vs FullAttention).

In [ ]:
for h in [24, 48, 96, 192]:
    print(f"\n==================== FINE-TUNE 12 TRẠM H={h} ====================")
    !python fine_tune.py --config configs/default.yaml --horizon {h} --models lstm dlinear informer patchtst --seed 42

print("\n==================== ABLATION STUDIES ====================")
!python run_ablation.py --config configs/default.yaml

## Cell 6 — Backup Tự động lên Google Drive (`1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL`)

- Xuất file `mlflow_runs_export.csv` khớp 100% cấu trúc sheet `Raw_MLflow_Logs`.
- Nén toàn bộ `mlruns/`, `outputs/logs/`, `outputs/ablation/`, `outputs/reports/`, `outputs/checkpoints/` và `docs/` thành `dl_pm25_bj_backup_[YYMMDD]_[HHMM].zip`.
- Đồng bộ sang thư mục Drive đã mount **và** đẩy trực tiếp vào Google Drive Folder ID `1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL` thông qua Google Drive API v3 (`google.colab.auth`).

In [ ]:
import glob
import shutil
from pathlib import Path

!python -m src.utils.exp_backup --mlruns-dir mlruns --output-dir outputs --backup-dir backups --include-checkpoints

backup_zips = sorted(Path("backups").glob("dl_pm25_bj_backup_*.zip"))
latest_zip = backup_zips[-1]
csv_export = Path("outputs/mlflow_runs_export.csv")

# 1. Lưu bản sao dự phòng vào Google Drive đã mount
shutil.copy2(latest_zip, DRIVE_BACKUP_DIR / latest_zip.name)
if csv_export.exists():
    shutil.copy2(csv_export, DRIVE_BACKUP_DIR / csv_export.name)
print(f"[DRIVE MOUNT] Đã đồng bộ {latest_zip.name} và mlflow_runs_export.csv tới {DRIVE_BACKUP_DIR}")

# 2. Upload trực tiếp lên Google Drive Folder ID: 1AaKHRC-_WUQ5eZSdY9nRcINfQkfcefJL
try:
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    auth.authenticate_user()
    drive_service = build("drive", "v3")

    for upload_file, mime in [(latest_zip, "application/zip"), (csv_export, "text/csv")]:
        if not upload_file.exists():
            continue
        file_metadata = {"name": upload_file.name, "parents": [GDRIVE_FOLDER_ID]}
        media = MediaFileUpload(str(upload_file), mimetype=mime, resumable=True)
        created = drive_service.files().create(body=file_metadata, media_body=media, fields="id").execute()
        print(f"[GDRIVE API] Đã upload {upload_file.name} -> file_id={created.get('id')} vào folder {GDRIVE_FOLDER_ID}")
except Exception as exc:
    print(f"[WARN] Không upload được qua Google Drive API ({exc}). Bản sao vẫn an toàn tại {DRIVE_BACKUP_DIR}.")